# Week 06 Pandas 병합과 연결 실습

이번 실습은 실제 월별 매출 엑셀 파일을 읽어 `concat()`으로 연결하고, 보조 메타데이터를 `merge()`로 결합하는 흐름으로 구성합니다. 연결과 병합 이후에는 `matplotlib`으로 요약 결과를 시각화해 검증합니다. 마지막에는 교수님 제공 추가 코드인 gradient 시각화 예제를 함께 정리합니다.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

## 0. 실습 데이터 경로 확인

- `sales-jan-2014.xlsx`
- `sales-feb-2014.xlsx`
- `sales-mar-2014.xlsx`

세 파일은 같은 열 구조를 가진 월별 매출 데이터이므로, 먼저 `concat()` 실습에 적합합니다.

In [ ]:
data_dir = Path('StudyNote/week06/data')
sales_files = {
    '2014-01': data_dir / 'sales-jan-2014.xlsx',
    '2014-02': data_dir / 'sales-feb-2014.xlsx',
    '2014-03': data_dir / 'sales-mar-2014.xlsx',
}

sales_files

## 1. 월별 엑셀 파일 읽기

In [ ]:
monthly_frames = {
    month: pd.read_excel(path)
    for month, path in sales_files.items()
}

for month, frame in monthly_frames.items():
    print(month, frame.shape)

monthly_frames['2014-01'].head()

## 2. `concat()`으로 3개월 매출 데이터 연결

같은 열 구조를 가진 월별 파일이므로 `axis=0` 방향 연결이 자연스럽습니다. 월 구분을 위해 `month` 열도 추가합니다.

In [ ]:
sales_all = pd.concat(
    [frame.assign(month=month) for month, frame in monthly_frames.items()],
    axis=0,
    ignore_index=True
)

sales_all.head()

## 3. 연결 결과 점검

연결 후에는 행 수, 열 이름, 자료형을 먼저 확인하는 습관이 중요합니다.

In [ ]:
print('shape =', sales_all.shape)
print('\ncolumns =')
print(sales_all.columns.tolist())
print('\ndtypes =')
print(sales_all.dtypes)

## 4. 날짜와 매출 컬럼 정리

In [ ]:
sales_all['date'] = pd.to_datetime(sales_all['date'])
sales_all['ext price'] = pd.to_numeric(sales_all['ext price'])
sales_all['quantity'] = pd.to_numeric(sales_all['quantity'])

sales_all[['date', 'ext price', 'quantity']].head()

## 5. 월별 매출 요약

`concat()`으로 연결한 데이터는 이후 `groupby()`로 요약해서 보는 흐름이 자연스럽습니다.

In [ ]:
monthly_summary = sales_all.groupby('month', as_index=False).agg(
    order_count=('account number', 'size'),
    total_sales=('ext price', 'sum'),
    avg_sales=('ext price', 'mean')
)

monthly_summary

## 6. `matplotlib`으로 월별 매출 시각화

연결과 집계가 끝났다면 그래프로 결과를 확인하는 것이 좋습니다. 막대그래프는 월별 총매출 비교에, 선그래프는 흐름 확인에 적합합니다.

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.bar(monthly_summary['month'], monthly_summary['total_sales'], color='steelblue')
plt.title('Monthly Total Sales')
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
plt.plot(monthly_summary['month'], monthly_summary['avg_sales'], marker='o', color='darkorange')
plt.title('Monthly Average Sales')
plt.xlabel('Month')
plt.ylabel('Average Sales')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 고객별 누적 매출 보기

In [ ]:
customer_summary = sales_all.groupby(['account number', 'name'], as_index=False).agg(
    total_quantity=('quantity', 'sum'),
    total_sales=('ext price', 'sum')
).sort_values('total_sales', ascending=False)

customer_summary.head(10)

## 8. 상위 고객 매출 시각화

In [ ]:
top_customers = customer_summary.head(10).sort_values('total_sales')

plt.figure(figsize=(8, 5))
plt.barh(top_customers['name'], top_customers['total_sales'], color='seagreen')
plt.title('Top 10 Customers by Total Sales')
plt.xlabel('Total Sales')
plt.ylabel('Customer')
plt.tight_layout()
plt.show()

## 9. `merge()` 실습용 고객 메타데이터 만들기

이제 실제 매출 데이터에 추가 정보를 붙이는 상황을 가정합니다. `account number`를 키로 해서 지역과 고객군 정보를 결합합니다.

In [ ]:
account_meta = pd.DataFrame({
    'account number': [740150, 412290, 218895, 527099, 999999],
    'region': ['East', 'West', 'East', 'South', 'North'],
    'segment': ['Enterprise', 'SMB', 'SMB', 'Enterprise', 'Public']
})

account_meta

## 10. `merge()`로 매출 요약과 메타데이터 결합

왼쪽 표를 기준으로 모두 남기기 위해 `left join`을 사용합니다.

In [ ]:
customer_with_meta = pd.merge(
    customer_summary,
    account_meta,
    on='account number',
    how='left'
)

customer_with_meta.head(10)

## 11. `indicator=True`로 병합 결과 확인

어느 고객이 메타데이터와 연결되지 않았는지 `_merge` 열로 점검할 수 있습니다.

In [ ]:
merge_checked = pd.merge(
    customer_summary,
    account_meta,
    on='account number',
    how='outer',
    indicator=True
)

merge_checked.head(15)

## 12. 추가 제공 코드 1: 2D 등고선과 gradient 벡터

아래 코드는 교수님이 함께 제공한 별도 실습 코드입니다. `Pandas` 병합과 직접 같은 주제는 아니지만, 함수값과 기울기 벡터를 시각화하는 예제로 따로 정리합니다.

In [ ]:
def f(w0, w1):
    return w0**2 + 2 * w0 * w1 + 3

def df_dw0(w0, w1):
    return 2 * w0 + 2 * w1

def df_dw1(w0, w1):
    return 2 * w0 + 0 * w1

w_range = 2
dw = 0.25
w0 = np.arange(-w_range, w_range + dw, dw)
w1 = np.arange(-w_range, w_range + dw, dw)
wn = w0.shape[0]

ww0, ww1 = np.meshgrid(w0, w1)
ff = np.zeros((len(w0), len(w1)))
dff_dw0 = np.zeros((len(w0), len(w1)))
dff_dw1 = np.zeros((len(w0), len(w1)))

for i0 in range(wn):
    for i1 in range(wn):
        ff[i1, i0] = f(w0[i0], w1[i1])
        dff_dw0[i1, i0] = df_dw0(w0[i0], w1[i1])
        dff_dw1[i1, i0] = df_dw1(w0[i0], w1[i1])

plt.figure(figsize=(9, 4))
plt.subplots_adjust(wspace=0.3)

plt.subplot(1, 2, 1)
cont = plt.contour(ww0, ww1, ff, 10, colors='k')
cont.clabel(fmt='%2.0f', fontsize=8)
plt.xticks(range(-w_range, w_range + 1, 1))
plt.yticks(range(-w_range, w_range + 1, 1))
plt.xlim(-w_range - 0.5, w_range + 0.5)
plt.ylim(-w_range - 0.5, w_range + 0.5)
plt.xlabel('$w_0$', fontsize=14)
plt.ylabel('$w_1$', fontsize=14)

plt.subplot(1, 2, 2)
plt.quiver(ww0, ww1, dff_dw0, dff_dw1)
plt.xlabel('$w_0$', fontsize=14)
plt.ylabel('$w_1$', fontsize=14)
plt.xticks(range(-w_range, w_range + 1, 1))
plt.yticks(range(-w_range, w_range + 1, 1))
plt.xlim(-w_range - 0.5, w_range + 0.5)
plt.ylim(-w_range - 0.5, w_range + 0.5)
plt.show()

## 13. 추가 제공 코드 2: 3D Surface와 Gradient Quiver

In [ ]:
ff = f(ww0, ww1)
dff_dw0 = df_dw0(ww0, ww1)
dff_dw1 = df_dw1(ww0, ww1)
dz = np.zeros_like(ff)

fig = plt.figure(figsize=(12, 6))
plt.subplots_adjust(wspace=0.1)

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(ww0, ww1, ff, rstride=1, cstride=1, cmap='viridis', alpha=0.8)
ax1.set_xlabel('$w_0$')
ax1.set_ylabel('$w_1$')
ax1.set_zlabel('$f(w_0, w_1)$')
ax1.set_title('3D Surface Plot')
fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=10)

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.quiver(ww0, ww1, ff, dff_dw0, dff_dw1, dz, length=0.2, normalize=True, color='red')
ax2.plot_wireframe(ww0, ww1, ff, rstride=2, cstride=2, linewidth=0.5, color='gray', alpha=0.3)
ax2.set_xlabel('$w_0$')
ax2.set_ylabel('$w_1$')
ax2.set_zlabel('$f(w_0, w_1)$')
ax2.set_title('3D Gradient (Quiver)')

plt.show()

## 14. 실습 체크 포인트

- 월별 엑셀처럼 같은 구조의 데이터는 `pd.concat()`으로 연결한다.
- 연결 뒤에는 `shape`, `columns`, `dtypes`를 먼저 점검한다.
- `groupby()` 결과는 `matplotlib`의 막대그래프와 선그래프로 빠르게 검증할 수 있다.
- 고객 추가 정보처럼 별도 표를 붙일 때는 `pd.merge()`를 사용한다.
- `indicator=True`는 병합 누락을 검증할 때 유용하다.
- gradient 시각화 코드는 6주차 핵심 주제와 별도이지만, 교수님 추가 코드로 함께 정리해 둘 수 있다.